## Small notebook to call/submit COMPAS processed consecutively 

In [7]:
import os
import re
import subprocess
from ruamel.yaml import YAML
import shutil
import json

# get the home dir and current working dir
home = os.path.expanduser("~")
cwd = os.getcwd()

## Flags used to run a population of WDWD systems
(with AIS)

   --include-WD-binaries-as-DCO: True&emsp;&emsp;&emsp;# Default: False (record WD binaries in the BSE DCO file)<br>
   <br>
   --add-options-to-sysparms: 'NEVER'&emsp;&emsp;&emsp;&emsp;# Default: 'GRID' # Options: ['NEVER','GRID','ALWAYS']<br>
   --logfile-definitions: 'COMPAS_Output_Definitions.txt'&emsp;&emsp;&emsp;&emsp;# Default: ''<br>


In [8]:
###############################################
###
###############################################
def replaceFileLine(file_dir, find_str, replace_str):
    """
    file_dir   = The file of which you would like to change a line
    find_str   = The line number that you want to change
    replace_str = The string that you want to replace this line wtth
    """
    #Open file of interest
    with open (file_dir, "r") as myfile:
        data = myfile.readlines()
    # Find the line that you want to change and replace it
    new_data = [re.sub(find_str, replace_str, line) for line in data]    
    # Write everything back
    with open (file_dir, "w") as wfile:
        wfile.writelines(new_data)

In [9]:
### Step 1: Change output dirs:  
new_dir=f"{home}/ceph/CompasOutput/v03.21.00/N5e6_NSNS_fid"

#stroopwafel_interface_Lieke.py,
replaceFileLine(f"{cwd}/MainRun/stroopwafel_interface_Lieke.py", find_str = r"output_folder = '.*'", 
                replace_str = f"output_folder = '{new_dir}/MainRun/'")

# COMPAS_r.sbatch
replaceFileLine(f"{cwd}/slurms/COMPAS_r.sbatch", find_str = r"data_dir=.*", 
                replace_str = f"data_dir='{new_dir}'")
# COMPAS_PP.sbatch 
replaceFileLine(f"{cwd}/slurms/COMPAS_PP.sbatch", find_str = r"data_dir=.*", 
                replace_str = f"data_dir='{new_dir}'")
# CosmicIntegration.sbatch
replaceFileLine(f"{cwd}/slurms/CosmicIntegration.sbatch", find_str = r"data_dir=.*", 
                replace_str = f"data_dir='{new_dir}'")


In [4]:
# Step 2: Submit COMPAS_r.sbatch (Stroopwafel run) and get the PID of the submitted job
try:
    result = subprocess.run(['sbatch', '--parsable', f'{cwd}/slurms/COMPAS_r.sbatch'], capture_output=True, text=True, check=True)
    SW_pid = result.stdout.strip()
    print(f"Submitted job PID: {SW_pid}")
except subprocess.CalledProcessError as e:
    print(f"Error submitting job: {e.stderr}")


Submitted job PID: 6572804


In [5]:
# Step 3: Submit COMPAS_PP.sbatch (postprocessing) after the completed run
try:
    result = subprocess.run(['sbatch', '--parsable', '--kill-on-invalid-dep=yes', f'--dependency=afterok:{SW_pid}', f'{cwd}/slurms/COMPAS_PP.sbatch'], capture_output=True, text=True, check=True) # 
    PP_pid = result.stdout.strip()
    print(f"Submitted job PID: {PP_pid}")
except subprocess.CalledProcessError as e:
    print(f"Error submitting job: {e.stderr}")


Submitted job PID: 6572805


In [6]:
# Step 4: Submit CosmicIntegration after postprocessing completed
# 
try:
    result = subprocess.run(['sbatch', '--parsable', '--kill-on-invalid-dep=yes', f'--dependency=afterok:{PP_pid}', f'{cwd}/slurms/CosmicIntegration.sbatch'], capture_output=True, text=True, check=True)
    CI_pid = result.stdout.strip()
    print(f"Submitted job PID: {CI_pid}")
except subprocess.CalledProcessError as e:
    print(f"Error submitting job: {e.stderr}")


Submitted job PID: 6572806


# Want to do it all in one go? 

## I.e. when you are looping over different variations


## Variations to explore:

   --common-envelope-allow-immediate-RLOF-post-CE-survive: True&emsp;&emsp;# Default: False<br>



In [10]:
# Run Variation dict:
flag_variants = [
    # === Fiducial ===
    {"simname": "fid",
        "overrides": {}
    },
    # === 5.1 Angular Momentum Loss Variations ===
    {"simname": "AM_MACLEOD_LINEAR",
        "overrides": {"--mass-transfer-angular-momentum-loss-prescription": "MACLEOD_LINEAR",
                      "--mass-transfer-jloss-macleod-linear-fraction-degen": 0.5,
                      "--mass-transfer-jloss-macleod-linear-fraction-non-degen": 0.5}
    },
    {"simname": "AM_circumbinary",
        "overrides": {"--mass-transfer-angular-momentum-loss-prescription": "CIRCUMBINARY"}
    },
    # === 5.2 Common Envelope Variations ===
    # CE lambda variations
    {"simname": "CE_lambda_Kruckow",
        "overrides": {"--common-envelope-lambda-prescription": "LAMBDA_KRUCKOW"}
    },
    # prescription from Loveridge et al., 2011;
    {"simname": "CE_lambda_Loveridge",
        "overrides": {"--common-envelope-lambda-prescription": "LAMBDA_LOVERIDGE"}
    },
    #``LAMBDA_DEWI`` is the fit from Appendix A in Claeys et al. 2014, based on Dewi & Tauris 2000
    {"simname": "CE_lambda_Dewi",
        "overrides": {"--common-envelope-lambda-prescription": "LAMBDA_DEWI"}
    },
    # CE alpha variations
    {"simname": "CE_alpha025",
        "overrides": {"--common-envelope-alpha": 0.25}
    },
    {"simname": "CE_alpha05",
        "overrides": {"--common-envelope-alpha": 0.5}
    },
    {"simname": "CE_alpha075",
        "overrides": {"--common-envelope-alpha": 0.75}
    },
    {"simname": "CE_alpha2",
        "overrides": {"--common-envelope-alpha": 2.0}
    },
    #  Hirai & Mandel (2022)
    {"simname": "CE_FORMALISM_TWO_STAGE",
        "overrides": {"--common-envelope-formalism": "TWO_STAGE"} 
    },
    # Various CE survival options
    {"simname": "CE_survive_radiative",
        "overrides": {"--common-envelope-allow-radiative-envelope-survive": True}
    },
    {"simname": "CE_survive_mainseq",
        "overrides": {"--common-envelope-allow-main-sequence-survive": True}
    },
    {"simname": "CE_postRLOF_survive",
        "overrides": {"--common-envelope-allow-immediate-RLOF-post-CE-survive": True}
    },
    # === 5.3 Mass Changes ===
    {"simname": "MS_accretor_mass_BRCEK",
        "overrides": {"--main-sequence-core-mass-prescription": "BRCEK"} # Core mass according to Shikauchi et al. (2024) with added rejuvenation of MS accretors and updated stellar tracks 
    },
    # Default = MANDEL The core following case A mass transfer is set equal to the expected core mass of a newly formed HG star with mass equal to that of the donor, scaled by the fraction of the donor's MS lifetime at mass transfer 
    {"simname": "remnant_FRYER2022",
        "overrides": {"--remnant-mass-prescription": "FRYER2022", 
        "--fryer-22-fmix": 0.5,
        "--fryer-22-mcrit": 5.75,
        "--kick-magnitude-distribution": "MAXWELLIAN"}
    },
    # === 5.4 Mass Transfer ===
    {"simname": "caseBB_AS_OTHER_MT",
        "overrides": {"--case-BB-stability-prescription": "TREAT_AS_OTHER_MT"}
    },
    {"simname": "MT_fixed_0.5",
        "overrides": {"--mass-transfer-accretion-efficiency-prescription": "FIXED",
            "--mass-transfer-fa": 0.5}
    },
    # === 5.5 Neutron Star Kicks ===
    {"simname": "Disberg_Mandel2025_kicks",
        "overrides": {"--kick-magnitude-distribution": "LOGNORMAL"}
        # ``LOGNORMAL`` applies kicks drawn from the Disberg & Mandel 2025 log-normal distribution.
    },
]

# Save the variations to a JSON file for reference
# with open("simulation_variations.json", "w") as f:
    # json.dump(flag_variants, f, indent=2)

In [15]:
# === CONFIG ===
script_src = "/mnt/home/lvanson/WDWD_NSNS_rates/code/COMPASsubmitscripts/masterfolder"
slurm_scripts = ["COMPAS_r.sbatch", "COMPAS_PP.sbatch", "CosmicIntegration.sbatch"]


In [16]:
# === CONFIG FILE OVERRIDER ===
def update_config_file(file_path, overrides):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        for flag, value in overrides.items():
            flag_escaped = re.escape(flag)
            pattern = rf"^(\s*)#?\s*{flag_escaped}:\s*.*"
            formatted = f"'{value}'" if isinstance(value, str) else "true" if value is True else "false" if value is False else str(value)

            match = re.match(pattern, line)
            if match:
                lines[i] = f"   {flag}: {formatted}\n"
                break

    for flag in overrides:
        if not any(re.match(rf"^\s*{re.escape(flag)}:", l) for l in lines):
            raise ValueError(f"Flag '{flag}' not found in config file!")

    with open(file_path, 'w') as f:
        f.writelines(lines)


In [18]:
###############################################
### Create all the dirs and files needed for a full run
### + submit the SLURM jobs (if not dryrun)
##################################################
def submit_full_run(simname, overrides, create_dirs_and_simdata=True, run_stroopwafel=True, run_PP=True, run_CI=True ):
    """
    Submits a full run for a given simulation variation.
    simname: Name of the simulation variation (used for directory naming).
    overrides: Dictionary of config overrides for this variation.
    create_dirs_and_simdata: If True, creates directories and copies scripts/config.
    run_stroopwafel: If True, submits the stroopwafel (COMPAS) job.
    run_PP: If True, submits the PostProcessing job.
    run_CI: If True, submits the CosmicIntegration job.
    """
    global home, cwd

    mainrun_dir = os.path.join(cwd, "MainRun")
    new_dir = f"{home}/ceph/CompasOutput/v03.21.00/N5e6_MassiveWDWD_NSNS_{simname}"
    post_dir = os.path.join(new_dir, "postProcessing")
    cosmic_dir = os.path.join(new_dir, "CosmicIntegration")
    mainrun_copy_dir = os.path.join(new_dir, "MainRun")

    if create_dirs_and_simdata:
        # Update stroopwafel interface with output path
        replaceFileLine(
            f"{mainrun_dir}/stroopwafel_interface_Lieke.py",
            find_str=r"output_folder = '.*'",
            replace_str=f"output_folder = '{new_dir}/MainRun/'"
        )

        # Patch slurm scripts with new data_dir
        for script in slurm_scripts:
            replaceFileLine(
                f"{cwd}/slurms/{script}",
                find_str=r"data_dir=.*",
                replace_str=f"data_dir='{new_dir}'"
            )

        # Copy config and apply overrides
        base_config_path = os.path.join(cwd, "compasConfig_fid.yaml")
        dest_config_path = os.path.join(mainrun_dir, "compasConfig.yaml")
        shutil.copy(base_config_path, dest_config_path)
        update_config_file(dest_config_path, overrides)

        # Create folders
        os.makedirs(post_dir, exist_ok=True)
        os.makedirs(cosmic_dir, exist_ok=True)
        os.makedirs(mainrun_copy_dir, exist_ok=True)

        # Copy scripts
        shutil.copy(os.path.join(script_src, "postProcessing", "h5copy.py"), post_dir)
        shutil.copy(os.path.join(script_src, "postProcessing", "append_weights.py"), post_dir)

        for fname in os.listdir(os.path.join(script_src, "CosmicIntegration")):
            shutil.copy(os.path.join(script_src, "CosmicIntegration", fname), cosmic_dir)

        shutil.copytree(os.path.join(script_src, "MainRun"), mainrun_copy_dir, dirs_exist_ok=True)

        shutil.copy(os.path.join(script_src, "slurms", "COMPAS_r.sbatch"), mainrun_copy_dir)
        shutil.copy(os.path.join(script_src, "slurms", "COMPAS_PP.sbatch"), post_dir)
        shutil.copy(os.path.join(script_src, "slurms", "CosmicIntegration.sbatch"), cosmic_dir)

        print(f"[INFO] Setup completed for '{simname}'.")

    # Paths to the per-run SLURM scripts
    run_script = os.path.join(mainrun_copy_dir, "COMPAS_r.sbatch")
    post_script = os.path.join(post_dir, "COMPAS_PP.sbatch")
    ci_script = os.path.join(cosmic_dir, "CosmicIntegration.sbatch")

    run_pid = None
    post_pid = None
    ci_pid = None

    if run_stroopwafel:
        run_proc = subprocess.run(["sbatch", "--parsable", run_script], capture_output=True, text=True)
        run_pid = run_proc.stdout.strip()
        print(f"Submitted COMPAS run for '{simname}' → job ID {run_pid}")
    else:
        print(f"[SKIPPED] No COMPAS run (stroopwafel) submitted")

    if run_PP:
        dep = [] if not run_pid else [f"--dependency=afterok:{run_pid}"]
        post_proc = subprocess.run(["sbatch", "--parsable"] + dep + [post_script], capture_output=True, text=True)
        post_pid = post_proc.stdout.strip()
        print(f"Submitted PostProcessing for '{simname}' → job ID {post_pid}")
    else:
        print(f"[SKIPPED] No PostProcessing submitted")

    if run_CI:
        dep = [] if not post_pid else [f"--dependency=afterok:{post_pid}"]
        ci_proc = subprocess.run(["sbatch"] + dep + [ci_script], capture_output=True, text=True)
        ci_pid = ci_proc.stdout.strip()
        print(f"Submitted CosmicIntegration for '{simname}' → job ID {ci_pid}")
    else:
        print(f"[SKIPPED] No CosmicIntegration submitted")

    return run_pid, post_pid, ci_pid


In [19]:
# submit loop (option to only run the PostProcessing and CosmicIntegration steps for all variations)
for i, variant in enumerate(flag_variants):
    # Only run the first variation (fiducial)
    if i != 0:
        break
    print(f"\n--- Preparing {variant['simname']} ---")
    submit_full_run(simname=variant["simname"], overrides=variant["overrides"], create_dirs_and_simdata=True, run_stroopwafel=True, run_PP=True, run_CI=True)




--- Preparing fid ---
[INFO] Setup completed for 'fid'.
Submitted COMPAS run for 'fid' → job ID 6572870
Submitted PostProcessing for 'fid' → job ID 6572871
Submitted CosmicIntegration for 'fid' → job ID Submitted batch job 6572872
